In [ ]:
# ─────────────────────────────────────────────────────────
#  Notebook  : Source ► Bronze (con manejo de eliminaciones)
#  Contexto  : Fabric Lakehouses
# ─────────────────────────────────────────────────────────

from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import max, col
import sys
import time
import random

# ------------------- 0. Parámetros y Asunciones Clave -------------------
log_table_path = f"Tables/QAD/UPDT_LOG"
log_source_table_column = "TABLE_NAME"
log_operation_type_column = "OPERATION_TYPE"
log_timestamp_column = "UPDATED_AT"
log_company_code_column = "COMPANY_CODE"
log_record_id_column = "RECORD_ID"

# Parámetro para controlar si esta tabla maneja eliminaciones desde el log
handles_deletes = True 

# Configuración para LEER fechas antiguas de fuentes Parquet/Delta
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
# Configuración para ESCRIBIR fechas antiguas a destinos Parquet/Delta (RECOMENDADA)
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# ───────── Rutas absolutas (para Fabric) ────────────────
staging_path = f"Tables/QAD/{table_name_temp}"           # I/U
dest_path    = f"Tables/QAD/{table_name}"

field_ID = load_column                                   
company_code = f"{table_name.split('_')[0]}_DOMAIN"
if table_name == "UPDT_LOG":
    company_code = "COMPANY_CODE"

table_to_update = "lh_control_erp.dbo.source_to_bronze_control"
update_condition = f"source_table = '{table_name}' AND source_system = '{source_system}' AND target_layer = '{target_layer}'"

# ───────── Control de concurrencia para actualización de tabla de control ─────────
MAX_RETRIES = 10
BASE_DELAY_SEC = 3
MAX_DELAY_SEC = 40

def is_concurrent_error(ex: Exception) -> bool:
    """Detecta si el error es por concurrencia en Delta Lake."""
    msg = str(ex).lower()
    return "concurrentappendexception" in msg or ("concurrent" in msg and "delta" in msg)

def run_control_update(update_query: str) -> None:
    """Ejecuta un UPDATE sobre la tabla de control con reintentos ante concurrencia."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            spark.sql(update_query)
            if attempt > 1:
                print(f"   Tabla de control actualizada (intento {attempt}).")
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                raise
    if last_error is not None:
        raise last_error

# ───────── Función utilitaria ───────────────────────────
def delta_table_exists(path: str) -> bool:
    """
    Devuelve True solo si la ruta corresponde a una tabla Delta existente.
    Si la ruta no existe o no es Delta, devuelve False.
    """
    try:
        DeltaTable.forPath(spark, path)
        return True
    except AnalysisException:
        return False

# ───────── Lógica principal ─────────────────────────────
try:
    # ------------------- 0.1 Leer info de control -------------------
    print("📥 Obteniendo información desde la tabla de control...")
    control_df = spark.sql(f"""
        SELECT last_watermark_value, last_success_run_at
        FROM {table_to_update}
        WHERE {update_condition}
    """)

    last_watermark_value = None
    last_success_run_at = None

    if control_df.count() > 0:
        row = control_df.collect()[0]
        last_watermark_value = row["last_watermark_value"]
        last_success_run_at = row["last_success_run_at"]
        print(f"   last_watermark_value : {last_watermark_value}")
        print(f"   last_success_run_at  : {last_success_run_at}")
    else:
        print("⚠️ No se encontró registro en la tabla de control. Se asume PRIMERA CARGA.")

    # ------------------- 0.2 Validar existencia de tabla destino -------------------
    dest_is_delta = delta_table_exists(dest_path)
    if dest_is_delta:
        print(f"✅ La tabla destino '{dest_path}' existe y es Delta.")
    else:
        print(f"ℹ️ La tabla destino '{dest_path}' NO existe o NO es Delta.")
        print("ℹ️ Se tratará como primera carga / recreación desde staging.")

    # ------------------- 1. Leer Staging (Nuevos y Actualizados) -------------------
    print("🔄 Cargando datos de staging (nuevos y actualizados)...")
    df_staging_raw = spark.read.format("delta").load(staging_path)
    df_staging = df_staging_raw.dropDuplicates([field_ID, company_code])
    count_staging = df_staging.count()
    print(f"📊 Registros en staging (deduplicados): {count_staging}")

    # ------------------- 2. Gestionar Eliminaciones (si aplica) -------------------
    # Solo aplica si:
    #  - handles_deletes = True
    #  - la tabla destino YA existe y es Delta
    #  - existe una ejecución exitosa previa (last_success_run_at no nulo)
    if handles_deletes:
        if dest_is_delta and last_success_run_at is not None:
            print("🔎 Buscando registros para eliminar en Bronze...")

            # Leer la tabla de log desde Bronze
            df_log = spark.read.format("delta").load(log_table_path)

            # Filtrar para obtener los IDs a eliminar posteriores a la última ejecución
            df_deletes = df_log.filter(
                (col(log_source_table_column) == table_name) &
                (col(log_operation_type_column) == 'D') &
                (col(log_timestamp_column) >= last_success_run_at)
            ).select(
                log_record_id_column,
                log_company_code_column
            ).dropDuplicates()

            count_deletes = df_deletes.count()
            print(f"🗑️ Se encontraron {count_deletes} registros para eliminar.")

            if count_deletes > 0:
                print("⚙️ Ejecutando MERGE para eliminaciones...")
                table_des_for_delete = DeltaTable.forPath(spark, dest_path)
                table_des_for_delete.alias("target").merge(
                    df_deletes.alias("deletes"),
                    f"target.{field_ID} = deletes.{log_record_id_column} "
                    f"AND target.{company_code} = deletes.{log_company_code_column}"
                ).whenMatchedDelete().execute()
                print("✅ Eliminaciones completadas.")
            else:
                print("ℹ️ No se encontraron registros para eliminar en esta corrida.")
        elif not dest_is_delta:
            print("ℹ️ Se omite manejo de eliminaciones porque la tabla destino todavía no existe como Delta.")
        elif last_success_run_at is None:
            print("ℹ️ Se omite manejo de eliminaciones porque no hay ejecución previa exitosa registrada (primera carga).")

    # ------------------- 3. MERGE para Inserciones y Actualizaciones -------------------
    if count_staging == 0:
        if not dest_is_delta:
            # NUEVO: crear tabla destino vacía si no existe y no hay datos en staging
            print("⚠️ No hay registros nuevos o actualizados en staging y la tabla destino no existe como Delta.")
            print("📁 Creando tabla destino vacía con el esquema de staging...")
            (
                df_staging_raw
                .limit(0)  # solo el esquema, sin filas
                .write
                .format("delta")
                .mode("overwrite")
                .save(dest_path)
            )
            print("✅ Tabla destino vacía creada en Bronze.")
        else:
            print("⚠️ No hay registros nuevos o actualizados en staging. Se omitirá el MERGE de inserción/actualización.")
    else:
        if not dest_is_delta:
            # Aquí se crea / recrea Bronze con datos
            print("📁 La tabla destino no existe como Delta. Creándola/recreándola con los datos de staging...")
            df_staging.write.format("delta").mode("overwrite").save(dest_path)
            print("✅ Tabla destino creada/recreada.")
        else:
            print("⚙️ Ejecutando MERGE para inserciones y actualizaciones...")
            table_des = DeltaTable.forPath(spark, dest_path)
            table_des.alias("target").merge(
                df_staging.alias("source"),
                f"target.{field_ID} = source.{field_ID} AND target.{company_code} = source.{company_code}"
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
            print("✅ MERGE de inserciones/actualizaciones completado.")

    # ------------------- 4. Actualizar Tabla de Control -------------------
    if count_staging > 0:
        max_wm = df_staging.agg(max(field_ID).cast("long").alias("max_wm")).collect()[0]["max_wm"]
        run_control_update(f"""
            UPDATE {table_to_update} SET
                last_watermark_value = '{max_wm}',
                last_run_status      = 'Success',
                last_run_at          = current_timestamp(),
                last_message         = 'Load successful',
                last_success_run_at  = current_timestamp()
            WHERE {update_condition}
        """)
        print(f"📝 Tabla de control actualizada a watermark = {max_wm}")
    else:
        run_control_update(f"""
            UPDATE {table_to_update} SET
                last_run_status     = 'Success',
                last_run_at         = current_timestamp(),
                last_message        = 'No new or updated records found',
                last_success_run_at = current_timestamp()
            WHERE {update_condition}
        """)
        print("📝 Tabla de control actualizada (sin nuevos registros).")

    # ------------------- 5. Vaciar Staging -------------------
    if table_name != "UPDT_LOG":
        print("🧹 Limpiando tabla staging (datos en Delta)...")
        spark.sql(f"DELETE FROM delta.`{staging_path}`")
        print("✅ Tabla staging vaciada.")

        # NUEVO: eliminar también la tabla temporal del catálogo QAD
        try:
            print("🧹 Eliminando tabla temporal de staging en el catálogo (QAD)...")
            spark.sql(f"DROP TABLE IF EXISTS QAD.{table_name_temp}")
            print(f"✅ Tabla staging QAD.{table_name_temp} eliminada del catálogo.")
        except Exception as e_drop:
            print(f"⚠️ No se pudo eliminar la tabla QAD.{table_name_temp}: {str(e_drop)}")

    print("🏁 Proceso finalizado correctamente.")

except Exception as e:
    print(f"!!!!!! ERROR en la carga de '{table_name}': {str(e)} !!!!!!", file=sys.stderr)
    err_msg = str(e).replace("'", "''")
    run_control_update(f"""
        UPDATE {table_to_update} SET
            last_run_status = 'Failed',
            last_run_at     = current_timestamp(),
            last_message    = '{err_msg}'
        WHERE {update_condition}
    """)
    raise e
